In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.1 Eigenvalues, Eigenvectors, and Diagonalization

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume III — Eigenvalues and Spectral Theory",
    number="3.1",
    title="Eigenvalues, Eigenvectors, and Diagonalization",
    blurb="The directions a matrix does not turn, the coordinates in which it "
    "becomes a list of numbers, and the matrix that has only one such direction "
    "when it needs two.",
    difficulty="intermediate",
    estimate="90–120 min",
)

## Notebook overview

An eigenvector is a direction the matrix does not turn. Along it, a linear map
— which in general shears, rotates and stretches all at once — does nothing
more interesting than multiply by a number. Find enough such directions to form
a basis and the map becomes, *in those coordinates*, a list of numbers: that is
diagonalization, and it is the point of the whole volume.

The payoff is immediate and computational. Powers of a diagonal matrix are
free, so $A^k = X\Lambda^kX^{-1}$ turns $k$ matrix multiplications into $k$
scalar powers, and every question about long-run behaviour — does this
iteration converge, does this system oscillate, what does the population do
after 30 generations — becomes a question about which $|\lambda_i|$ is largest.
We work that out explicitly on the Fibonacci matrix, where the dominant
eigenvalue is the golden ratio and diagonalization produces Binet's closed
form for $F_n$ out of nothing but a $2\times2$ matrix.

The notebook ends where the theory does. Diagonalization requires $n$
independent eigenvectors and a matrix need not have them. The $2\times2$ matrix
$\left[\begin{smallmatrix}1&1\\0&1\end{smallmatrix}\right]$ has the eigenvalue
1 twice and only *one* eigenvector, so no $X$ exists, and `numpy` will hand
back a matrix of eigenvectors whose rank is 1 without saying a word about it.
Recognising that failure is what [§3.5](schur-jordan-nonnormality.ipynb) is for.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own. This notebook leans on
> that idea harder than most, because an eigenvector is only defined up to
> scale: two correct answers can differ by a factor of $-1$ and a check that
> does not know this will call one of them wrong.

> **Scope.** Strang {cite}`strang2023` Chapter 6 and Axler {cite}`axler2024`
> Chapter 5 for the theory (Axler builds it without determinants, which is
> worth seeing); Trefethen and Bau {cite}`trefethen1997` Lectures 24–25 for the
> numerical side; Golub and Van Loan {cite}`golub2013` Chapter 7.

## Theory in brief

### The definition, and what it asks for

A scalar $\lambda$ and a **nonzero** vector $\mathbf{x}$ satisfying

```{math}
:label: eq-eig-def
A\mathbf{x} = \lambda\mathbf{x}
```

are an eigenvalue and an eigenvector of $A$. The requirement $\mathbf{x}\ne\mathbf{0}$
is what gives the definition content, since $\mathbf{0}$ satisfies it for every
$\lambda$. Rewriting as $(A - \lambda I)\mathbf{x} = \mathbf{0}$ says
$\mathbf{x}$ lies in the null space of $A - \lambda I$, and by
[§1.3](../01-matrices/inverses-rank-cr.ipynb) a nonzero null space means a
singular matrix, so

```{math}
:label: eq-eig-charpoly
\det(A - \lambda I) = 0 .
```

The left side is a degree-$n$ polynomial in $\lambda$, the **characteristic
polynomial**, and its $n$ roots (over $\mathbb{C}$, with multiplicity) are the
eigenvalues. Two warnings come with it. Real matrices can have complex
eigenvalues — a rotation turns *every* real direction, so it has no real
eigenvector, and its eigenvalues are $\pm i$. And {eq}`eq-eig-charpoly` is a
fine *definition* and a terrible *algorithm*: polynomial root-finding is badly
conditioned, and no serious library computes eigenvalues this way
([§5.2](../05-numerical/eigenvalue-algorithms.ipynb) explains what they do
instead).

**Eigenvectors are directions, not vectors.** If $A\mathbf{x} = \lambda\mathbf{x}$
then $A(c\mathbf{x}) = \lambda(c\mathbf{x})$ for every $c \ne 0$. Any check that
compares eigenvector *components* against expected numbers is therefore
checking a convention, not a fact.

### Two invariants that come free

Expanding {eq}`eq-eig-charpoly` and matching coefficients gives

```{math}
:label: eq-eig-invariants
\operatorname{tr} A = \sum_{i=1}^{n}\lambda_i ,
\qquad
\det A = \prod_{i=1}^{n}\lambda_i .
```

Both hold with complex eigenvalues, with repeated ones, and for matrices that
cannot be diagonalized at all. They are the cheapest available check on any
computed spectrum, and this notebook uses them as one.

### Diagonalization

Suppose $A$ has $n$ *linearly independent* eigenvectors. Put them in the
columns of $X$ and the eigenvalues on the diagonal of $\Lambda$. Then
$AX = X\Lambda$ column by column, and $X$ is invertible precisely because the
columns are independent, so

```{math}
:label: eq-eig-diag
A = X\Lambda X^{-1} .
```

This is a **change of basis** in the sense of
[§1.6](../01-matrices/linear-maps-change-of-basis.ipynb): $X^{-1}$ expresses a
vector in the eigenbasis, $\Lambda$ scales each coordinate, $X$ converts back.
The map was always a list of scalings; the standard basis was hiding it.

The immediate consequence is that powers telescope, since the inner $X^{-1}X$
pairs cancel:

```{math}
:label: eq-eig-power
A^k = X\Lambda^kX^{-1} ,
\qquad \Lambda^k = \operatorname{diag}(\lambda_1^k, \dots, \lambda_n^k) .
```

For large $k$ the term with the largest $|\lambda_i|$ dominates everything
else, which is why the **spectral radius** $\rho(A) = \max_i|\lambda_i|$
decides whether $A^k$ grows or decays.

### Binet's formula, as an application

The Fibonacci recurrence $F_{n+1} = F_n + F_{n-1}$ is
$\left[\begin{smallmatrix}F_{n+1}\\F_n\end{smallmatrix}\right] =
F\left[\begin{smallmatrix}F_n\\F_{n-1}\end{smallmatrix}\right]$ with
$F = \left[\begin{smallmatrix}1&1\\1&0\end{smallmatrix}\right]$, whose
eigenvalues are the golden ratio $\varphi = (1+\sqrt5)/2$ and
$\psi = (1-\sqrt5)/2$. Applying {eq}`eq-eig-power` and reading off one entry
gives

```{math}
:label: eq-eig-binet
F_n = \frac{\varphi^n - \psi^n}{\sqrt5} ,
```

a closed form for an integer sequence, built from two irrational numbers, out
of a $2\times2$ diagonalization.

### Power iteration

{eq}`eq-eig-power` also suggests an algorithm. Repeatedly applying $A$ and
renormalising,

```{math}
:label: eq-eig-poweriter
\mathbf{x}_{k+1} = \frac{A\mathbf{x}_k}{\|A\mathbf{x}_k\|_2} ,
```

amplifies the dominant eigendirection relative to every other one by
$|\lambda_1/\lambda_2|$ per step, so $\mathbf{x}_k$ converges to the dominant
eigenvector at the linear rate $|\lambda_2/\lambda_1|^k$ — for *almost every*
starting vector, the exception being those with no component along
$\mathbf{v}_1$ at all. The eigenvalue is then recovered by the **Rayleigh
quotient**

```{math}
:label: eq-eig-rayleigh
\rho(\mathbf{x}) = \frac{\mathbf{x}^{\top}\!A\mathbf{x}}{\mathbf{x}^{\top}\mathbf{x}} .
```

### When it all fails

Independence of the eigenvectors is an assumption, not a theorem. Call the
multiplicity of $\lambda$ as a root of {eq}`eq-eig-charpoly` its **algebraic**
multiplicity and $\dim\ker(A - \lambda I)$ its **geometric** multiplicity. The
geometric never exceeds the algebraic, and when it is strictly smaller the
matrix is **defective**: it has fewer than $n$ independent eigenvectors, no $X$
exists, and {eq}`eq-eig-diag` is simply false for it.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

from ecp import animate, validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps
np.set_printoptions(precision=6, suppress=True, linewidth=110)

# The notebook's three worked matrices, all integer and all fixed here.
A3 = np.array([[2.0, 0.0, 0.0],
               [-1.0, 3.0, 1.0],
               [-2.0, 2.0, 2.0]])          # non-symmetric, eigenvalues 1, 2, 4
A2 = np.array([[3.0, 2.0],
               [1.0, 2.0]])                # eigenvalues 4, 1
A_POW = np.array([[6.0, 2.0],
                  [-1.0, 3.0]])            # eigenvalues 5, 4: a slow power iteration
FIB = np.array([[1.0, 1.0],
                [1.0, 0.0]])               # the Fibonacci matrix
DEFECTIVE = np.array([[1.0, 1.0],
                      [0.0, 1.0]])         # one eigenvalue, one eigenvector


def eig_residual(A, w, V):
    """Largest ||A x - lambda x|| over all returned eigenpairs.

    The one check on an eigendecomposition that is basis-independent and
    convention-free: it tests Eq. 1 itself rather than any particular scaling,
    sign or ordering of the vectors a library chose to return.
    """
    return max(float(np.linalg.norm(A @ V[:, i] - w[i] * V[:, i]))
               for i in range(len(w)))

## Exercise 1 — The characteristic polynomial, exactly and then numerically

{eq}`eq-eig-charpoly` reduces the eigenvalue problem to root-finding, and for a
small integer matrix SymPy can carry that out over the rationals, giving
eigenvalues that are exactly right rather than approximately so. That exact
answer is then the yardstick for what `np.linalg.eig` returns.

The matrix is

$$
A_3 = \begin{bmatrix} 2 & 0 & 0\\ -1 & 3 & 1\\ -2 & 2 & 2 \end{bmatrix},
$$

available as `A3` above. It is deliberately **not** symmetric, so nothing in
this notebook can quietly borrow the guarantees of
[§3.2](spectral-theorem.ipynb).

**Part a)** Build `sp.Matrix(A3.astype(int).tolist())` and compute the
characteristic polynomial with `.charpoly(lam).as_expr()` for
`lam = sp.symbols("lamda")`. Print it, then factor it with `sp.factor`. It
comes out $(\lambda-1)(\lambda-2)(\lambda-4)$, so the eigenvalues are exactly
$1, 2, 4$.

**Part b)** Get the exact eigenvectors with `.eigenvects()`, which returns
triples `(eigenvalue, algebraic multiplicity, [basis of the eigenspace])`.
Report all three. Each multiplicity is 1 and each eigenspace is one
dimensional, so $A_3$ has three independent eigenvectors and is diagonalizable.

**Part c)** Now the numerical route: `w, V = np.linalg.eig(A3)`. Confirm the
returned eigenvalues match the exact $\{1, 2, 4\}$ to $10^{-13}$ **after
sorting**, since `eig` promises no particular order. Report the ordering it
actually returned.

**Part d)** Confirm {eq}`eq-eig-def` itself for every returned pair with
`eig_residual(A3, w, V)`, which comes out below $10^{-15}$. This is the check
to trust: it tests the defining equation and nothing about the convention.

**Part e)** Note the caveat about method. Root-finding on
{eq}`eq-eig-charpoly` is how eigenvalues are *defined* and not how they are
*computed*: the roots of a polynomial can be wildly sensitive to its
coefficients, so forming the characteristic polynomial and solving it is
numerically unsound above small sizes. Confirm `np.roots` on the polynomial's
coefficients agrees here to $10^{-13}$ — at $n = 3$ with integer entries the
route is harmless — while noting that
[§5.2](../05-numerical/eigenvalue-algorithms.ipynb) is where the real
algorithm lives.

In [ ]:
# (solution hidden on the public site)


### Validation 1

The eigenvalues are compared *sorted*, because `eig` guarantees no ordering,
and the eigenvectors are checked only through the residual of
{eq}`eq-eig-def`, because their scaling and sign are the library's to choose.
Both of those are the never-gate-on-a-library-choice rule applied.

In [ ]:
validate.close(
    np.sort(w3.real), exact,
    "np.linalg.eig reproduces the exact eigenvalues 1, 2, 4 (Eq. 2)",
    rtol=0.0, atol=1e-13,
)
validate.close(
    np.abs(w3.imag), np.zeros(3),
    "and returns them real, as the exact factorization over Q says they are",
    rtol=0.0, atol=1e-14,
)
validate.check(
    eig_residual(A3, w3, V3) < 1e-14,
    "every returned pair satisfies the defining equation A x = lambda x (Eq. 1)",
    f"largest residual {eig_residual(A3, w3, V3):.3e} over the three pairs",
)
validate.check(
    all(len(vecs) == mult for _, mult, vecs in A3s.eigenvects()),
    "geometric multiplicity equals algebraic for all three: A_3 IS diagonalizable",
    "each eigenvalue is simple and contributes exactly one independent "
    "eigenvector, so the three together span R^3",
)
validate.close(
    roots_np, exact,
    "and np.roots on the characteristic polynomial agrees here (Eq. 2)",
    rtol=0.0, atol=1e-13,
)

## Exercise 2 — What `eig` is free to choose, and what it is not

The previous exercise checked eigenvalues by sorting and eigenvectors by
residual, and both evasions were deliberate. This exercise makes the reason
explicit, because it is the single most common way an otherwise correct
eigenvalue computation gets marked wrong.

{eq}`eq-eig-def` is homogeneous in $\mathbf{x}$: if $\mathbf{x}$ is an
eigenvector then so is $c\mathbf{x}$ for every nonzero scalar $c$, with the
*same* eigenvalue. An eigenvector is therefore a **one-dimensional subspace**,
and any particular vector returned is one representative of it chosen by a
convention. LAPACK's convention is unit 2-norm, but the sign — and for complex
eigenvectors the phase — is whatever the algorithm happened to produce.

**Part a)** Confirm the convention: report `np.linalg.norm(V3[:, i])` for
$i = 0, 1, 2$ and check each is 1 to $10^{-14}$.

**Part b)** Confirm the freedom. For the first returned eigenvector
$\mathbf{v}_0$ and each of $c = -1$, $2.5$, $-0.3$, verify that $c\mathbf{v}_0$
satisfies {eq}`eq-eig-def` with the same eigenvalue, to $10^{-14}$. Nothing
distinguishes these from the returned one except the norm.

**Part c)** Show what a naive check would do. Compare the returned
eigenvector for $\lambda = 2$ against the exact $(1, 1, 0)$ from Exercise 1
*directly*, entry by entry, and report the discrepancy — it is large, because
the exact vector is not normalised and the sign may differ. Then compare them
the right way, as directions, via
$|\cos\theta| = |\mathbf{u}^{\top}\mathbf{v}|/(\|\mathbf{u}\|\|\mathbf{v}\|)$,
and confirm it is 1 to $10^{-14}$: the two are the same direction.

**Part d)** State the alternative that is always safe. Rather than comparing
eigenvectors at all, compare the **projector** onto the eigenspace,
$P_i = \mathbf{v}_i\mathbf{v}_i^{\top}/(\mathbf{v}_i^{\top}\mathbf{v}_i)$,
which is independent of scale and sign entirely. Build it from the returned
vector and from the exact one and confirm they agree to $10^{-14}$.

**Part e)** Confirm the ordering is also unconstrained: report the order `eig`
returned for `A3` against the sorted order, and confirm that sorting by
`np.argsort(w.real)` reorders `V3`'s **columns** consistently, so that
`eig_residual` still passes after the permutation.

In [ ]:
# (solution hidden on the public site)


### Validation 2

The failing comparison is run deliberately and reported as a number rather
than hidden, because seeing the naive check fail on a *correct* answer is the
point of the exercise. The projector comparison is the one to reach for
whenever an eigenvector really must be checked against a reference.

In [ ]:
validate.close(
    norms, np.ones(3),
    "eig returns unit-norm eigenvectors — a convention, not a theorem",
    rtol=0.0, atol=1e-14,
)
validate.check(
    max(scaled_res) < 1e-14,
    "and every rescaling c*v is equally an eigenvector, same lambda (Eq. 1)",
    f"largest residual {max(scaled_res):.2e} over c = -1, 2.5, -0.3: an "
    "eigenvector is a DIRECTION, so its components carry no information a check "
    "may rely on",
)
validate.check(
    entrywise > 0.1,
    "so an entrywise comparison against a reference vector fails on a correct answer",
    f"max |difference| = {entrywise:.4f} against the exact (1, 1, 0), purely "
    "because that vector is not normalised",
)
validate.close(
    np.array([cosine]), np.array([1.0]),
    "while the same two vectors are the same direction to 1e-14",
    rtol=0.0, atol=1e-14,
)
validate.close(
    P_num, P_exact,
    "and the eigenspace projector, immune to scale and sign, matches outright",
    rtol=0.0, atol=1e-14,
)
validate.check(
    eig_residual(A3, w_sorted, V_sorted) < 1e-14,
    "sorting is safe only if the eigenvector columns are permuted with the values",
    f"residual {eig_residual(A3, w_sorted, V_sorted):.2e} after applying "
    f"argsort {order} to both",
)

## Exercise 3 — Two invariants, and a matrix with no real eigenvector

{eq}`eq-eig-invariants` gives two scalars computable directly from the entries
of $A$, in $O(n)$ and $O(n^3)$ respectively, that must agree with quantities
built from the spectrum. They are the cheapest sanity check on a computed
eigendecomposition and they hold in complete generality: complex eigenvalues,
repeated eigenvalues, defective matrices, all of it.

The second matrix here is the $90^\circ$ rotation
$R = \left[\begin{smallmatrix}0&-1\\1&0\end{smallmatrix}\right]$, which is the
standard demonstration that a real matrix need have no real eigenvector. A
rotation turns every direction in the plane, so no real $\mathbf{x}$ can
satisfy {eq}`eq-eig-def`; the eigenvalues are $\pm i$ and the eigenvectors are
complex.

**Part a)** For `A3`, report $\operatorname{tr}A_3$ from `np.trace` and
$\sum\lambda_i$ from `w3.sum()`, and confirm they agree to $10^{-13}$. Both are
7.

**Part b)** Report $\det A_3$ from `np.linalg.det` and $\prod\lambda_i$ from
`np.prod(w3)`, and confirm agreement to $10^{-13}$. Both are 8, and
$1\cdot2\cdot4 = 8$ exactly.

**Part c)** Build `R = np.array([[0.0, -1.0], [1.0, 0.0]])` and take
`np.linalg.eig(R)`. Report the eigenvalues, confirm they are $\pm i$ to
$10^{-15}$, and confirm $|\lambda| = 1$ for both — a rotation changes no
length, so it cannot scale any direction by anything other than a unit-modulus
number.

**Part d)** Confirm {eq}`eq-eig-invariants` still holds for $R$: the trace is
0 and $i + (-i) = 0$; the determinant is 1 and $i \cdot (-i) = 1$. Check both
to $10^{-15}$. The identities did not need real eigenvalues.

**Part e)** Confirm no real eigenvector exists, by checking that the imaginary
parts of `R`'s eigenvectors are not negligible: report
`np.abs(V_R.imag).max()` and confirm it exceeds $0.5$. There is nothing wrong
with the matrix; the eigenvectors simply live in $\mathbb{C}^2$, which is what
[§3.4](hermitian-unitary-normal.ipynb) is about.

In [ ]:
# (solution hidden on the public site)


### Validation 3

The invariants are checked on both matrices, and the rotation is the more
informative of the two: it confirms {eq}`eq-eig-invariants` survives complex
eigenvalues, which is the case where a reader might reasonably doubt it.

In [ ]:
validate.check(
    tr_gap < 1e-13 and det_gap < 1e-13,
    "trace = sum of eigenvalues and det = product, for A_3 (Eq. 3)",
    f"trace gap {tr_gap:.2e}, determinant gap {det_gap:.2e}; 1 + 2 + 4 = 7 and "
    "1 * 2 * 4 = 8",
)
validate.close(
    np.sort_complex(wR), np.array([-1j, 1j]),
    "the 90-degree rotation has eigenvalues exactly +/- i (Eq. 2)",
    rtol=0.0, atol=1e-15,
)
validate.close(
    np.abs(wR), np.ones(2),
    "both of modulus 1: a rotation cannot scale any direction",
    rtol=0.0, atol=1e-15,
)
validate.check(
    abs(np.trace(R) - wR.sum().real) < 1e-15
    and abs(np.linalg.det(R) - np.prod(wR).real) < 1e-15,
    "and the invariants of Eq. 3 hold for it too, complex eigenvalues and all",
    f"trace 0 against i + (-i) = {wR.sum().real:.1e}; det 1 against "
    f"i * (-i) = {np.prod(wR).real:.12f}",
)
validate.check(
    np.abs(VR.imag).max() > 0.5,
    "with genuinely complex eigenvectors: no real direction is invariant",
    f"largest |imaginary part| = {np.abs(VR.imag).max():.4f} — every real vector "
    "in the plane is turned by 90 degrees, so none can satisfy Eq. 1 over R",
)

## Exercise 4 — Diagonalization, and why powers become free

With three independent eigenvectors in hand, {eq}`eq-eig-diag` is available and
{eq}`eq-eig-power` follows. The practical content is that a matrix power costs
one eigendecomposition plus $n$ scalar powers, instead of $k-1$ matrix
multiplications at $2n^3$ flops each — and, more importantly, that the *answer*
becomes readable: $A^k$ is a fixed combination of projectors with the numbers
$\lambda_i^k$ in front, so its behaviour as $k$ grows is obvious by inspection.

**Part a)** Form `X = V3` and `Lam = np.diag(w3)` and reconstruct
$X\Lambda X^{-1}$ with `np.linalg.inv`. Confirm it reproduces `A3` to
$10^{-14}$, which is {eq}`eq-eig-diag` verified.

**Part b)** Report `np.linalg.cond(X)`, the conditioning of the eigenvector
matrix. It is about 4 here, which is small, and that number is what decides
whether {eq}`eq-eig-diag` is *numerically* useful as well as algebraically
true. Exercise 8 meets a matrix where it is $10^{16}$.

**Part c)** Compute $A_3^{10}$ two ways: as
`(X @ np.diag(w3**10) @ np.linalg.inv(X)).real` and as
`np.linalg.matrix_power(A3, 10)`. Report the largest entry of the result
($7.0\times10^{5}$), the largest absolute difference ($1.2\times10^{-10}$), and
the difference *relative to the matrix scale* ($1.7\times10^{-16}$). Check the
relative form to $10^{-13}$, not the absolute one: a bare $10^{-12}$ on entries
of size $10^{5}$ would be demanding 21 significant digits.

**Part d)** Read the long-run behaviour off the spectrum. With eigenvalues
$1, 2, 4$, the spectral radius is 4, so $\|A_3^k\|$ should grow like $4^k$.
Compute `np.linalg.norm(np.linalg.matrix_power(A3, k), 2)` for
$k = 4, 6, 8, 10, 12$, fit a straight line to $\log\|A^k\|$ against $k$ with
`np.polyfit`, and confirm the slope is $\log 4$ to within 3%.

**Part e)** Confirm the eigenvalues of $A_3^{10}$ are the tenth powers of the
eigenvalues of $A_3$ — that is, $\{1, 1024, 1048576\}$ — to a *relative*
$10^{-12}$, which is {eq}`eq-eig-power` read as a statement about spectra.

In [ ]:
# (solution hidden on the public site)


### Validation 4

The $A^{10}$ comparison is gated *relative to the matrix scale*, following the
rule [§0.1](../00-machine/arrays-and-vectorization.ipynb) established: entries
of size $10^{5}$ carry absolute rounding of order $10^{-11}$, so an absolute
tolerance below that would be demanding accuracy the format cannot hold.

In [ ]:
validate.close(
    (X @ Lam @ Xinv).real, A3,
    "A = X Lambda X^-1 reconstructs the matrix exactly (Eq. 4)",
    rtol=0.0, atol=1e-14,
)
validate.check(
    abs_gap / scale < 1e-13,
    "A^10 by diagonalization matches matrix_power, relative to the matrix scale",
    f"absolute gap {abs_gap:.2e} on entries of size {scale:.2e}, i.e. relative "
    f"{abs_gap/scale:.2e} — an absolute 1e-12 here would demand 21 digits",
)
validate.close(
    np.array([slope]), np.array([np.log(4.0)]),
    "||A^k|| grows like rho(A)^k = 4^k, from the fitted log-slope (Eq. 5)",
    rtol=3e-2, atol=0.0,
)
validate.close(
    w10, np.array([1.0, 1024.0, 1048576.0]),
    "and the eigenvalues of A^10 are the 10th powers of those of A (Eq. 5)",
    rtol=1e-12, atol=0.0,
)
validate.check(
    np.linalg.cond(X) < 10.0,
    "with cond(X) small, so the diagonalization is usable and not just true",
    f"cond(X) = {np.linalg.cond(X):.4f}; Exercise 8 meets a matrix where this "
    "number is 1e16 and Eq. 4 stops meaning anything",
)

## Exercise 5 — Fibonacci, the golden ratio, and Binet's formula

The Fibonacci recurrence is a linear map applied repeatedly, so
{eq}`eq-eig-power` applies and produces a closed form. Writing
$\mathbf{u}_n = \left[\begin{smallmatrix}F_{n+1}\\F_n\end{smallmatrix}\right]$,
the recurrence is $\mathbf{u}_n = F\mathbf{u}_{n-1}$ with
$F = \left[\begin{smallmatrix}1&1\\1&0\end{smallmatrix}\right]$, hence
$\mathbf{u}_n = F^n\mathbf{u}_0$ — and $F^n$ is diagonalizable, so this can be
written out explicitly and gives {eq}`eq-eig-binet`.

**Part a)** Take `w_f, V_f = np.linalg.eig(FIB)` and confirm the eigenvalues
are $\varphi = (1+\sqrt5)/2 = 1.6180339887$ and $\psi = (1-\sqrt5)/2 =
-0.6180339887$ to $10^{-14}$, after sorting descending.

**Part b)** Confirm the two invariants of {eq}`eq-eig-invariants` in the form
they take here: $\varphi + \psi = \operatorname{tr}F = 1$ and
$\varphi\psi = \det F = -1$, each to $10^{-15}$. The second is the defining
property of the golden ratio, appearing as a determinant.

**Part c)** Evaluate Binet's formula {eq}`eq-eig-binet` for
$n = 10, 20, 30, 40, 50$ and compare against the exact integers from
`sympy.fibonacci(n)`. Report the absolute error for each. It grows from
$1.4\times10^{-14}$ to $1.9\times10^{-5}$, and $F_{30} = 832040$ is reproduced
to $8\times10^{-10}$.

**Part d)** Explain the growth, and confirm the explanation. The *relative*
error stays at machine level throughout — check that it is below $10^{-14}$ for
every $n$ tested — while the absolute error grows because $F_n$ itself grows
like $\varphi^n$. Binet's formula is not becoming less accurate; the numbers
are becoming larger. Note also what this means in practice: an exact-integer
sequence computed through irrational arithmetic can never return an exact
integer, so `round()` is doing real work.

**Part e)** Confirm the matrix route agrees: `np.linalg.matrix_power(FIB, n)`
has $F_n$ in its off-diagonal entry, exactly for $n \le 50$ since these
integers are below $2^{53}$. Check `matrix_power(FIB, 30)[0, 1]` equals
$832040$ exactly, and confirm the same for $n = 50$ against
$12586269025$.

In [ ]:
# (solution hidden on the public site)


### Validation 5

Binet is checked *both* ways round on purpose: absolutely, where it visibly
degrades, and relatively, where it does not. Reporting only the first would
suggest a formula going wrong; reporting only the second would hide that
$F_{50}$ cannot be recovered as an integer this way.

In [ ]:
validate.close(
    w_f_sorted, np.array([phi, psi]),
    "the Fibonacci matrix has eigenvalues phi and psi (Eq. 2)",
    rtol=0.0, atol=1e-14,
)
validate.check(
    abs(phi + psi - np.trace(FIB)) < 1e-15
    and abs(phi * psi - np.linalg.det(FIB)) < 1e-15,
    "with phi + psi = tr F = 1 and phi psi = det F = -1 (Eq. 3)",
    f"sum {phi + psi:.15f}, product {phi * psi:.15f}: the golden ratio's "
    "defining property, appearing as a determinant",
)
validate.close(
    np.array([(phi**30 - psi**30) / np.sqrt(5.0)]), np.array([832040.0]),
    "Binet's formula reproduces F_30 = 832040 (Eq. 6)",
    rtol=0.0, atol=1e-8,
)
validate.check(
    max(binet_rel) < 1e-14 and binet_abs[-1] > 1e-6,
    "and its RELATIVE accuracy holds while the absolute error grows with F_n",
    f"worst relative {max(binet_rel):.2e} against absolute {binet_abs[-1]:.2e} "
    f"at n = 50: an integer sequence computed through sqrt(5) never returns an "
    "integer",
)
validate.check(
    round(float(F30[0, 1])) == 832040 and round(float(F50[0, 1])) == 12586269025,
    "while repeated matrix multiplication stays exact below 2^53",
    f"matrix_power gives {F30[0,1]:.0f} and {F50[0,1]:.0f}, both exact integers, "
    "because every intermediate is a whole number a float64 can hold",
)

## Exercise 6 — Invariant directions, and why they are not the ellipse's axes

An eigenvector is a direction the matrix does not turn, and the cleanest way to
see that is to watch what $A$ does to the unit circle. The circle maps to an
ellipse, and almost every direction is rotated in the process; along an
eigenvector, and only there, the image points the same way as the input.

There is a second lesson available in the same picture, and it is worth taking
early. The ellipse has axes, and it is tempting to assume they are the
eigenvectors. They are not. The axes are the **singular** directions, the
subject of Volume IV, and for a non-symmetric matrix they differ from the
eigenvectors — and the axis *lengths* are the singular values, which differ
from the eigenvalues. The two coincide only when $A$ is symmetric, which is
exactly the content of [§3.2](spectral-theorem.ipynb).

The matrix is $A_2 = \left[\begin{smallmatrix}3&2\\1&2\end{smallmatrix}\right]$,
available as `A2`, with eigenvalues 4 and 1.

**Part a)** Take `w2, V2 = np.linalg.eig(A2)`, confirm the eigenvalues are 4
and 1 to $10^{-14}$ after sorting, and report the eigenvectors rescaled to
integer form. They are $(2, 1)$ for $\lambda = 4$ and $(-1, 1)$ for
$\lambda = 1$.

**Part b)** Confirm the invariance directly. For each eigenvector, compute
$|\cos\theta|$ between $\mathbf{v}$ and $A\mathbf{v}$ and confirm it is 1 to
$10^{-14}$: the image points along the same line. Then do the same for the
non-eigenvector $(1, 0)$ and report the angle in degrees, which is about
$18.4^\circ$ — a generic direction *is* turned.

**Part c)** Compute the singular values with
`np.linalg.svd(A2, compute_uv=False)` and report them beside the eigenvalues.
They are $4.1306$ and $0.9684$, against eigenvalues 4 and 1. Confirm they are
different, and confirm the one identity that *does* link them:
$\prod\sigma_i = |\det A| = \prod|\lambda_i| = 4$, to $10^{-13}$.

**Part d)** Confirm the eigenvectors are not orthogonal: report the angle
between them, which is $71.565^\circ$. For a symmetric matrix it would be
exactly $90^\circ$, and that is the whole difference the next notebook is
about.

**Part e)** Draw the unit circle and its image under $A_2$ with
`la.unit_circle_image`, and overlay the two eigendirections, so that the
invariant directions and the ellipse's axes can be seen to be different
objects.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

Invariance is checked as $|\cos\theta| = 1$ between $\mathbf{v}$ and
$A\mathbf{v}$, which is scale- and sign-free and is therefore a statement about
the *direction*, exactly as Exercise 2 argued. The eigenvalue–singular-value
distinction is checked by the one identity that genuinely connects them.

In [ ]:
validate.close(
    w2, np.array([4.0, 1.0]),
    "A_2 has eigenvalues 4 and 1 (Eq. 2)",
    rtol=0.0, atol=1e-14,
)
validate.close(
    np.array(inv_cos), np.ones(2),
    "and along each eigenvector the image points the same way (Eq. 1)",
    rtol=0.0, atol=1e-14,
)
validate.check(
    gen_cos < 0.98,
    "while a generic direction is genuinely turned",
    f"|cos| = {gen_cos:.6f} for (1, 0), that is "
    f"{np.degrees(np.arccos(gen_cos)):.2f} degrees of rotation",
)
validate.check(
    np.abs(svals - w2).max() > 0.05 and prod_gap < 1e-13,
    "singular values differ from eigenvalues, but their products agree",
    f"sigma = {np.array2string(svals, precision=4)} against lambda = "
    f"{np.array2string(w2, precision=4)}; both products equal |det A| = "
    f"{abs(np.linalg.det(A2)):.6f} to {prod_gap:.1e}",
)
validate.check(
    abs(eig_angle - 90.0) > 10.0,
    "and the eigenvectors are not orthogonal, because A_2 is not symmetric",
    f"they meet at {eig_angle:.3f} degrees; section 3.2 is the theorem that "
    "makes this exactly 90 when A = A^T",
)

## Exercise 7 — Power iteration, and watching a circle collapse to a line

{eq}`eq-eig-power` says that applying $A$ repeatedly multiplies the component
along $\mathbf{v}_i$ by $\lambda_i^k$. The component along the *dominant*
eigenvector therefore outgrows every other one, and after renormalising, what
survives is the dominant direction alone. That is {eq}`eq-eig-poweriter`, the
oldest eigenvalue algorithm there is, and the ancestor of both the Krylov
methods of Volume V and the PageRank computation of Volume VI.

The matrix is $A_{\text{pow}} =
\left[\begin{smallmatrix}6&2\\-1&3\end{smallmatrix}\right]$, available as
`A_POW`, with eigenvalues 5 and 4 and eigenvectors $(2,-1)$ and $(1,-1)$. The
ratio $|\lambda_2/\lambda_1| = 0.8$ is deliberately close to 1, so the
convergence is slow enough to watch.

**Part a)** Confirm the spectrum: `np.linalg.eig(A_POW)` gives 5 and 4 to
$10^{-14}$, so $\operatorname{tr} = 9$ and $\det = 20$.

**Part b)** Write `power_iteration(A, x0, n_steps)` implementing
{eq}`eq-eig-poweriter`, returning `(iterates, rayleigh)` where `iterates` is an
array of shape `(n_steps + 1, len(x0))` holding every normalised iterate and
`rayleigh` holds the Rayleigh quotient {eq}`eq-eig-rayleigh` at each step.
Normalise with `x / np.linalg.norm(x)` at every step.

**Write this one yourself** — the implementation is the lesson.

**Part c)** Run it from $\mathbf{x}_0 = (1, 0)$ for 40 steps. Report the
distance to the dominant eigendirection at $k = 0, 10, 20, 30, 40$, measured as
$\min(\|\mathbf{x}_k - \mathbf{v}_1\|, \|\mathbf{x}_k + \mathbf{v}_1\|)$ — the
minimum over the sign, because the iterate may converge to either
representative. It falls from $0.460$ to $2.7\times10^{-5}$.

**Part d)** Confirm the *rate*. Fit a straight line to $\log(\text{error})$
against $k$ over steps 8 to 30 with `np.polyfit` and confirm the slope is
$\log|\lambda_2/\lambda_1| = \log 0.8 = -0.2231$ to within 10%. Gating the
fitted rate rather than an absolute error is the course's rule for measured
exponents.

**Part e)** Report the Rayleigh quotient at $k = 0, 10, 20, 40$ and confirm it
converges to $\lambda_1 = 5$, with $|\rho_{40} - 5| < 10^{-3}$. Note the rate:
the errors are $6.8\times10^{-2}$, $7.0\times10^{-3}$, $8.0\times10^{-5}$ at
$k = 10, 20, 40$, so $\rho$ converges at the same linear rate as the vector,
*not* faster. The famous quadratic convergence of the Rayleigh quotient
requires a symmetric matrix, and $A_{\text{pow}}$ is not one.

**Part f)** Animate the collapse. Take 24 starting directions evenly spaced
around the unit circle (offset by 0.07 radians so that none of them is exactly
the second eigenvector, which would be a fixed point), apply
{eq}`eq-eig-poweriter` to all of them simultaneously, and animate the resulting
points on the unit circle over 40 steps. Confirm from the data — not from the
animation — that after 40 steps every one of the 24 directions is within
$10^{-6}$ of the dominant eigendirection in $1 - |\cos\theta|$.

```{admonition} With your assistant
:class: tip
Power iteration finds the *largest* eigenvalue. **Inverse iteration** finds the
one nearest a chosen shift $\mu$, by applying $(A - \mu I)^{-1}$ instead — its
dominant eigenvalue is $1/(\lambda_{\text{nearest}} - \mu)$, so the same
argument converges to the eigenvector one actually wants. Ask your assistant to
write `inverse_iteration(A, mu, x0, n_steps)` using `scipy.linalg.lu_factor`
once and `lu_solve` inside the loop, never forming the inverse. Then check it
yourself against the mathematics: run it on `A_POW` with $\mu = 4.1$ and verify
the resulting vector satisfies $\|A\mathbf{x} - 4\mathbf{x}\| < 10^{-10}$ —
the exact defining equation, Eq. 1, for the eigenvalue power iteration cannot
reach. Then check that the convergence rate is
$|\lambda_1 - \mu|^{-1}/|\lambda_2 - \mu|^{-1}$ with the roles of 4 and 5
swapped. The check is yours.
```

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

The convergence *rate* is gated, not an absolute error, because the rate is the
theoretical prediction and the absolute error after a fixed number of steps is
not. The animation's validation checks the underlying data — the 24 directions
— and never the animation object, following the course's rule.

In [ ]:
validate.close(
    np.sort(w_p.real)[::-1], np.array([5.0, 4.0]),
    "A_pow has eigenvalues 5 and 4 (Eq. 2)",
    rtol=0.0, atol=1e-14,
)
validate.close(
    np.array([fit[0]]), np.array([np.log(ratio)]),
    "power iteration converges at the rate |lambda_2/lambda_1|^k (Eq. 7)",
    rtol=1e-1, atol=0.0,
)
validate.check(
    err[-1] < 1e-4 and err[-1] < err[0] / 1e4,
    "reaching the dominant eigendirection to 1e-4 after 40 steps",
    f"distance {err[-1]:.3e} at k = 40 against {err[0]:.3e} at k = 0",
)
validate.check(
    abs(rq[-1] - 5.0) < 1e-3,
    "and the Rayleigh quotient converges to lambda_1 = 5 (Eq. 8)",
    f"rho_40 = {rq[-1]:.8f}, error {abs(rq[-1] - 5.0):.2e}",
)
validate.check(
    abs(rq[20] - 5.0) / abs(rq[10] - 5.0) > 0.5 * ratio**10,
    "at the SAME linear rate as the vector, not the quadratic rate",
    f"|rho - 5| goes {abs(rq[10]-5):.2e} -> {abs(rq[20]-5):.2e} over ten steps, "
    f"a factor of {abs(rq[20]-5)/abs(rq[10]-5):.3f} against ratio^10 = "
    f"{ratio**10:.3f}; quadratic convergence needs A = A^T",
)
validate.check(
    spread[-1] < 1e-6,
    "and all 24 starting directions collapse onto the dominant eigenvector",
    f"1 - min|cos(angle to v_1)| = {spread[-1]:.3e} after {N_STEPS} steps, from "
    f"{spread[0]:.3e} at the start",
)

## Exercise 8 — The matrix that cannot be diagonalized

Everything above assumed $n$ independent eigenvectors. That assumption can
fail, and when it does, {eq}`eq-eig-diag` is not approximately true or
numerically delicate — it is false, because no invertible $X$ exists.

The standard example is
$J = \left[\begin{smallmatrix}1&1\\0&1\end{smallmatrix}\right]$, available as
`DEFECTIVE`. Being triangular, its eigenvalues are its diagonal entries, so
$\lambda = 1$ with algebraic multiplicity 2. But
$J - I = \left[\begin{smallmatrix}0&1\\0&0\end{smallmatrix}\right]$ has rank 1,
so its null space is one dimensional and the geometric multiplicity is 1. There
is one eigendirection where there ought to be two.

What makes this exercise worth doing is that `numpy` does not tell you. It
returns two eigenvalues and a $2\times2$ matrix of eigenvectors, and one has
to look to notice that the two columns are the same direction.

**Part a)** Confirm the eigenvalues from `np.linalg.eig(DEFECTIVE)` are both 1
to $10^{-15}$, and confirm the invariants: $\operatorname{tr}J = 2 = 1 + 1$ and
$\det J = 1 = 1\cdot1$. Nothing is wrong yet.

**Part b)** Print the returned eigenvector matrix `V_d`. Confirm its two
columns are the same direction up to sign, by checking $|\cos\theta| = 1$
between them to $10^{-14}$, and confirm `np.linalg.matrix_rank(V_d)` is 1, not
2. There is only one eigendirection, $(1, 0)$.

**Part c)** Report `np.linalg.cond(V_d)`. It is about $9\times10^{15}$, of
order $1/\varepsilon$ — the numerical signature of a defective matrix, and the
quantity Exercise 4 checked was small. A large `cond(X)` is the warning that
{eq}`eq-eig-diag` is about to stop meaning anything.

**Part d)** Confirm the algebraic diagnosis exactly. With
`Js = sp.Matrix([[1, 1], [0, 1]])`, check that `Js.eigenvects()` reports
algebraic multiplicity 2 with a **one**-element eigenspace basis, and that
`Js.diagonalize()` raises `sp.matrices.exceptions.MatrixError`. SymPy, working
exactly, refuses; `numpy`, working numerically, cannot.

**Part e)** See what replaces {eq}`eq-eig-power`. Compute $J^n$ for
$n = 2, 5, 10, 50$ with `np.linalg.matrix_power` and confirm the closed form
$J^n = \left[\begin{smallmatrix}1&n\\0&1\end{smallmatrix}\right]$ exactly. Then
note what it means: every eigenvalue has $|\lambda| = 1$, so the spectral
radius is exactly 1 and the naive reading of {eq}`eq-eig-power` predicts
bounded powers — yet $\|J^n\|$ grows **linearly** without bound. Confirm this
properly rather than at a single $n$, and exactly rather than by a fit: the
singular values of $J^n$ are the square roots of the eigenvalues of
$(J^n)^{\top}J^n = \left[\begin{smallmatrix}1&n\\n&n^2+1\end{smallmatrix}\right]$,
which has trace $n^2+2$ and determinant 1, giving

$$
\|J^n\|_2 = \frac{n + \sqrt{n^2+4}}{2} = n + \frac{1}{n} + O(n^{-3}) .
$$

Compute $\|J^n\|_2$ for $n = 10, 50, 100, 500, 1000$ and confirm this closed
form to a relative $10^{-12}$: growth is linear in $n$, with an explicit
constant. The
eigenvalues did not lie, but they did not tell the whole truth either, and that
gap is what [§3.5](schur-jordan-nonnormality.ipynb) is about.

In [ ]:
# (solution hidden on the public site)


### Validation 8

The rank of the eigenvector matrix is the check that matters, and it is an
integer, so there is no tolerance to argue about. The final check is the one
that motivates the rest of the volume: a matrix whose every eigenvalue sits on
the unit circle, whose powers nevertheless grow without bound.

In [ ]:
validate.close(
    w_d.real, np.ones(2),
    "the defective matrix has eigenvalue 1 twice (Eq. 2)",
    rtol=0.0, atol=1e-15,
)
validate.check(
    rank_Vd == 1,
    "but its eigenvector matrix has rank 1: only ONE independent eigendirection",
    f"|cos| between the two returned columns is {col_cos:.12f}, so they are the "
    "same direction; numpy returned a 2-by-2 matrix without any warning",
)
validate.check(
    cond_Vd > 1e14,
    "and cond(X) is of order 1/eps, the numerical signature of defectiveness",
    f"cond(V_d) = {cond_Vd:.3e} against 1/eps = {1/EPS:.2e}; Exercise 4's "
    f"cond(X) was {np.linalg.cond(X):.2f}, which is what a usable "
    "diagonalization looks like",
)
validate.check(
    alg == 2 and geo == 1 and diag_raised,
    "SymPy, working exactly, refuses to diagonalize it at all (Eq. 4)",
    f"algebraic multiplicity {alg} against geometric {geo}; diagonalize() "
    "raises MatrixError, because no invertible X exists",
)
validate.check(
    powers_ok,
    "J^n = [[1, n], [0, 1]] exactly, for n = 2, 5, 10, 50",
    "the superdiagonal entry counts the steps: this is what Eq. 5 becomes when "
    "the matrix cannot be diagonalized",
)
validate.check(
    max(abs(w_d)) == 1.0 and norm50 > 40.0,
    "so every |lambda| = 1 while ||J^50|| is about 50: bounded spectrum, "
    "unbounded powers",
    f"rho(J) = {max(abs(w_d)):.1f} exactly, ||J^50||_2 = {norm50:.4f}. Reading "
    "growth off the spectral radius alone is safe only in the limit, and "
    "section 3.5 is the honest account",
)
validate.close(
    norms_J, closed,
    "and ||J^n||_2 = (n + sqrt(n^2+4))/2 exactly: linear growth, closed form",
    rtol=1e-12, atol=0.0,
)

---
## Notebook summary

**The definition, and what a library may choose.** For the non-symmetric
integer matrix $A_3$, SymPy factored the characteristic polynomial exactly as
$(\lambda-1)(\lambda-2)(\lambda-4)$ and `np.linalg.eig` reproduced those roots
to $10^{-15}$ once sorted, with every returned pair satisfying
$A\mathbf{x} = \lambda\mathbf{x}$ to $6\times10^{-17}$. Eigenvectors came back
unit-norm by convention, and rescaling by $-1$, $2.5$ or $-0.3$ left
{eq}`eq-eig-def` satisfied to $10^{-16}$ — so an entrywise comparison against
the exact $(1,1,0)$ **failed by 0.29** on a perfectly correct answer, while the
direction cosine was 1 to $10^{-16}$ and the eigenspace projectors agreed to
$10^{-16}$.

**Invariants hold in every case.** $\operatorname{tr}A_3 = 7 = 1+2+4$ and
$\det A_3 = 8 = 1\cdot2\cdot4$; the $90^\circ$ rotation has eigenvalues exactly
$\pm i$, both of modulus 1, with trace $0 = i + (-i)$ and determinant
$1 = i\cdot(-i)$, and eigenvectors that are genuinely complex — no real
direction survives a rotation.

**Diagonalization makes powers free and behaviour legible.**
$X\Lambda X^{-1}$ reconstructed $A_3$ to $4\times10^{-16}$ with
$\operatorname{cond}(X) = 3.9$; $A_3^{10}$ agreed with `matrix_power` to a
*relative* $1.7\times10^{-16}$ on entries of size $7\times10^{5}$; and
$\|A_3^k\|$ grew with fitted log-slope matching $\log 4 = \log\rho(A_3)$ to
under 1%.

**Binet's formula, from a $2\times2$ diagonalization.** The Fibonacci matrix's
eigenvalues are $\varphi$ and $\psi$ with $\varphi+\psi = \operatorname{tr} = 1$
and $\varphi\psi = \det = -1$, and {eq}`eq-eig-binet` reproduced
$F_{30} = 832040$ to $8\times10^{-10}$. Its absolute error grows to
$1.9\times10^{-5}$ by $n = 50$ while the relative error stays below
$10^{-15}$: the formula is not decaying, $F_n$ is growing.

**Eigenvectors are invariant directions, and not the ellipse's axes.** Under
$A_2$, the two eigendirections satisfied $|\cos\theta| = 1$ between
$\mathbf{v}$ and $A\mathbf{v}$ to $10^{-16}$ while the generic direction
$(1,0)$ was turned by $18.4^\circ$. The singular values $4.131, 0.968$ differ
from the eigenvalues $4, 1$, though both products equal $|\det A| = 4$; and the
eigenvectors met at $71.6^\circ$, not a right angle, because $A_2$ is not
symmetric.

**Power iteration converges at the ratio of the top two eigenvalues.** From
$(1,0)$ on $A_{\text{pow}}$ (eigenvalues 5 and 4), the distance to the dominant
eigendirection fell from $0.460$ to $2.7\times10^{-5}$ in 40 steps, with fitted
log-slope matching $\log 0.8$ to under 2%. All 24 starting directions collapsed
to within $10^{-8}$. The Rayleigh quotient reached 5 with error
$8\times10^{-5}$, converging at the *same* linear rate as the vector — the
quadratic rate requires symmetry.

**And the theory has a hole.** $J = \left[\begin{smallmatrix}1&1\\0&1\end{smallmatrix}\right]$
has algebraic multiplicity 2 and geometric multiplicity 1. `numpy` returned two
eigenvalues and two eigenvectors that are the *same direction*, with
$\operatorname{rank} = 1$ and $\operatorname{cond}(X) = 9\times10^{15}$, and
said nothing; SymPy raised `MatrixError`. $J^n = \left[\begin{smallmatrix}1&n\\0&1\end{smallmatrix}\right]$
exactly, so $\rho(J) = 1$ while $\|J^{50}\|_2 > 50$.

**Methods introduced.** `np.linalg.eig` and `eigvals`, `sympy.Matrix.charpoly`,
`eigenvals`, `eigenvects` and `diagonalize`, `np.linalg.matrix_power`, the
eigenspace projector as a convention-free comparison, power iteration and the
Rayleigh quotient, and `np.linalg.cond(X)` as the diagnostic for whether a
diagonalization is usable.

## Outlook

- **When the eigenvectors are orthogonal.** Everything awkward in this notebook
  — non-orthogonal eigenvectors, a badly conditioned $X$, complex eigenvalues,
  a Rayleigh quotient that converges only linearly — disappears when
  $A = A^{\top}$. [§3.2](spectral-theorem.ipynb) proves it and collects the
  consequences, which are unreasonably good.
- **What to do about the defective case.** The Jordan form describes $J$
  perfectly and cannot be computed in floating point at all, since an
  arbitrarily small perturbation makes any matrix diagonalizable. The Schur
  decomposition $A = QTQ^{*}$ is the computable substitute, and it always
  exists. [§3.5](schur-jordan-nonnormality.ipynb) builds both, and the
  pseudospectra that explain why $\|J^{50}\| > 50$ was predictable.
- **How eigenvalues are actually computed.** Not by the characteristic
  polynomial. The QR algorithm reduces to Hessenberg form and then iterates,
  and [§5.2](../05-numerical/eigenvalue-algorithms.ipynb) is where that
  happens — along with the reason forming $\det(A - \lambda I)$ is a bad idea
  the moment $n$ is more than a handful.
- **Where power iteration goes next.** Keeping the *whole history* of iterates
  rather than just the last one gives the Krylov subspace, and extracting
  eigenvalues from it gives Arnoldi and Lanczos
  ([§5.5](../05-numerical/krylov-gmres-preconditioning.ipynb)). The same
  iteration applied to a stochastic matrix is PageRank
  ([§6.2](../06-structure/markov-perron-pagerank.ipynb)).

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()